# Entity Disambiguation

This is a very simple NER and entity disambiguation pipeline. People, places, organisations, and works or art are extracted from the text and then passed as a query to the appropriate query function in this library. Up to 5 candidates are returned which the model may select one from or reject them all. The notebook expects that you have a model served at an openai compatible endpoint on port `30002`. Alternatively, the notebook can be modified to point at OpenAI using your API key.

In [1]:
import asyncio
import json

from openai import AsyncClient

from national_gallery_api import NationalGallery, render_candidates

### Fetch Samples

Fetch descriptions for all objects that have them at the gallery as samples for tagging.

In [2]:
with NationalGallery(cache=True) as ng:
    descriptions = [work.description[0].value for work in ng.works.iter_all() if hasattr(work, "description")]

print(descriptions[0])

'While working on his meticulous paintings in the studio, Seurat also made small studies outdoors on wooden panels, which he called croquetons. He used these panels to record first-hand the atmospheric effects of nature, its play of light and colour, before transforming them into large-scale canvases using his rigorous pointillist method.\n\nThis study was made while the artist worked on A Sunday Afternoon on the Island of La Grande Jatte (1884-6, Art Institute of Chicago). Seurat paints the Seine River as a band running across the top of the panel, using distinctive dabs of blue and pink to suggest a gleaming surface, and explores the placement of an isolated figure on the greenery of the riverbank – a space that becomes populated with groups of leisurely middle- and upper-class Parisians in the final work. The sporadic dark shapes underneath the paint surface might indicate other figures Seurat initially tested out, before covering them up with his typical abbreviated marks of contra

### Prepare for Tagging

Prepare tagging prompts and messages for the model.

In [12]:
tagging_sys_prompt = {
    "role": "system",
    "content": """\
You are an expert at named-entity recognition for art-historical texts.

Extract every named entity from the text below and classify each into exactly one of these types:

- work_of_art: titles of paintings, sculptures, prints, drawings, series, or other artworks (e.g. "Sunflowers").
- person: named individuals — artists, sitters, patrons, historical figures (e.g. "Monet").
- location: geographic places — rivers, cities, regions, countries, buildings that are places (e.g. "London").
- organisation: institutions, galleries, museums, companies, groups (e.g. "Art Institute of Chicago").

Rules:
- Only extract entities that are explicitly named in the text. Do not infer or add entities that are not present.
- Use the exact surface form as it appears in the text; do not normalise, translate, or expand it.
- List each distinct entity once, even if it is mentioned multiple times.
- If a type has no entities, return an empty list for it.
- Respond with JSON only — no explanation, no markdown fences.

Return a JSON object with exactly these keys, each mapping to a list of strings:

{
  "work_of_art": [],
  "person": [],
  "location": [],
  "organisation": []
}
""",
}

_tag_template = """\
Tag the following text:
\"\"\"
{text}
\"\"\"
"""

# Create array of samples for model
tag_messages = [
    [tagging_sys_prompt, {"role": "user", "content": _tag_template.format(text=sample_text)}]
    for sample_text in descriptions
]

### Extract Entities

Setup client and generate function and run extraction over samples.

In [13]:
client = AsyncClient(base_url="http://localhost:30002/v1", api_key="none", timeout=60)
model_list = await client.models.list()

MODEL_NAME = model_list.data[0].id  # detect model name on expected port
NUM_CONCURRENT = 10  # number of simultaneous requests
MAX_SAMPLES = 10  # number of descriptions to tag
SAMPLING_PARAMS = {
    "temperature": 0.01  # Low temperature for better consistency
}


async def acompletion(messages: list[list[dict]], model: str | None = None, **request_kwargs) -> list[str]:
    """Async generate completions for faster batch processing"""
    semaphore = asyncio.Semaphore(NUM_CONCURRENT)
    total = len(messages)
    completed = 0

    async def _generate(message):
        nonlocal completed
        if not message:
            return ""

        try:
            async with semaphore:
                response = await client.chat.completions.parse(
                    messages=message, model=model or MODEL_NAME, **request_kwargs
                )
                return response.choices[0].message.content
        except Exception as e:
            print(f"An error occurred: {e}")
        finally:
            completed += 1
            print(f"\r{completed} / {total} complete", end="", flush=True)

    results = await asyncio.gather(*[_generate(message) for message in messages])
    return results


# select full dataset or subset for extraction
samples = tag_messages
if MAX_SAMPLES:
    samples = samples[:MAX_SAMPLES]

samples_tags = await acompletion(samples, **SAMPLING_PARAMS)

10 / 10 complete

### Disambiguate Entities

Create disambiguation prompt and prepare messages for model. For each extracted entity, query National Gallery API and render returned candidates for LLM disambiguation.

In [ ]:
disambiguation_prompt = {
    "role": "system",
    "content": """\
You are an expert at entity disambiguation (entity linking) for art-historical texts.

You will be given:
- A TARGET entity: a name mentioned in the source text, together with its type (work_of_art, person, location, or organisation).
- The SOURCE TEXT in which that name appears, to be used as context.
- Up to 5 CANDIDATE records retrieved from the National Gallery collection. Each candidate is introduced by a line "Candidate N:" followed by its type and title, a "PID:" field, and further distinguishing fields.

Your task is to decide which single candidate, if any, refers to the same real-world entity as the TARGET mention in the SOURCE TEXT.

Rules:
- Select a candidate only if you are confident it denotes the same entity as the target mention, judged from the context in the source text and the candidate's fields.
- A matching title or name alone is not sufficient — use the type, dates, roles, and other fields to confirm the match and to rule out same-name distractors.
- If none of the candidates is the correct entity, reject them all.
- Never select more than one candidate.

Respond with the PID of the selected candidate, copied exactly from that candidate's "PID:" field, and nothing else.
If no candidate is correct, respond with exactly:
None

Output only the PID string or the word None — no explanation, quotation marks, punctuation, or markdown.
""",
}

_disambig_template = """\
Source text:
\"\"\"
{text}
\"\"\"

Entity string: {entity_str}
Entity type: {entity_type}

Candidates:
{candidates}
"""


def parse_extracted_ents(output: str) -> dict:
    output = output.strip()
    try:
        return json.loads(output)
    except json.JSONDecodeError as e:
        print(f"Error parsing model output: {output}\n\n{e}")
        return {}


def iter_mentions(sources, extractions):
    # `sources` (all descriptions) is intentionally longer than `extractions` (only the first
    # MAX_SAMPLES were tagged), so zip stops at the shorter: strict=False is deliberate here.
    for source, extracted in zip(sources, extractions, strict=False):
        for entity_type, entity_strings in extracted.items():
            for entity_str in entity_strings:
                yield source, entity_type, entity_str


samples_jsons = [parse_extracted_ents(s) for s in samples_tags]
mentions = list(iter_mentions(descriptions, samples_jsons))

# build messages with rendered candidates from api queries
with NationalGallery(cache=True) as ng:
    search_map = {
        "work_of_art": ng.works,
        "person": ng.people,
        "location": ng.locations,
        "organisation": ng.organisations,
    }
    disambig_messages = [
        [
            disambiguation_prompt,
            {
                "role": "user",
                "content": _disambig_template.format(
                    text=source,
                    entity_str=entity_str,
                    entity_type=entity_type,
                    candidates=render_candidates(search_map[entity_type].search(entity_str, size=5)),
                ),
            },
        ]
        for source, entity_type, entity_str in mentions
    ]

# run disambiguation model
disambig_results = await acompletion(disambig_messages, **SAMPLING_PARAMS)

### Results

Parse each response (a PID or `None`) and map it back to the mention it came from.

In [ ]:
def parse_pid(output: str | None) -> str | None:
    """Normalise output to be PID or literal None."""
    output = (output or "").strip()
    return None if output.lower() in {"", "none"} else output


with NationalGallery(cache=True) as ng:
    search_map = {
        "work_of_art": ng.works,
        "person": ng.people,
        "location": ng.locations,
        "organisation": ng.organisations,
    }
    links = []
    for (_source, entity_type, entity_str), result in zip(mentions, disambig_results, strict=True):
        pid = parse_pid(result)
        title = search_map[entity_type].get(pid).title if pid else None
        links.append(
            {
                "entity_str": entity_str,
                "entity_type": entity_type,
                "pid": pid,
                "linked_title": title,
            }
        )

linked = [link for link in links if link["pid"]]
print(f"{len(linked)} / {len(links)} mentions linked to a collection record", end="\n\n")
for link in linked:
    print(f"\t{link['entity_type']:<13} {link['entity_str']!r}  ->  {link['pid']}  ({link['linked_title']})")

### Result with Source Text

Tagging is reasonably accurate but the model was given very minimal guidance in the task prompt. Entities that aren't grounded typically lack collections records in the Gallery database.

In [ ]:
first_text = descriptions[2]
print(first_text, end="\n\n")

print("Extracted entities:")
for (source, entity_type, entity_str), link in zip(mentions, links, strict=True):
    if source != first_text:
        continue
    if link["pid"]:
        print(f"\t[matched]   {entity_type:<13} {entity_str!r}  ->  {link['pid']}  ({link['linked_title']})")
    else:
        print(f"\t[no match]  {entity_type:<13} {entity_str!r}")

### Extending this Example

This notebook showed a very simple method for NER and disambiguation using the library but it is far from optimal. Extracting terms like this prevents multiple occurrences of the same string being tagged/disambiguated differently, there are lots of redundant LLM calls (e.g. where 0 candidates are returned; many of the disambiguations were likely trivial and didn't need a costly LLM call), and a smaller task-specific model could have been used for faster and probably better NER tagging.

The other issue is that only using the extracted string for search can result in very generic names such as in the example above with "Mary". This can be improved by asking the LLM to write query strings using the whole text rather than being limited to a contiguous span. The model could generate, for example, "Mary Gainsborough" (although in this instance it makes no difference as she doesn't appear in the collections records).

In [2]:
with NationalGallery(cache=True) as ng:
    results = ng.people.search("Mary Gainsborough", size=5)
    print(render_candidates(results))

Candidate 1:
Person: Mary Oswald
  PID: 02U9-0009-0000-0000
  Names: Mary Oswald; Mary Ramsay; Oswald, Mary
  Dates: 1719 - 1788
  Roles: subject
  External IDs: https://www.wikidata.org/entity/Q76196787; https://www.wikidata.org/entity/Q97379516

Candidate 2:
Person: Gainsborough Dupont
  PID: 052E-000B-0000-0000
  Names: Gainsborough Dupont; Dupont, Gainsborough
  Dates: 1754 - 1797
  Roles: publication author
  External IDs: http://viaf.org/viaf/10914963; http://vocab.getty.edu/ulan/500115199; https://rkd.nl/artists/24933; https://www.wikidata.org/entity/Q5517292

Candidate 3:
Person: Thomas Gainsborough
  PID: 0PFP-0001-0000-0000
  Names: Thomas Gainsborough; Gainsborough, Thomas
  Dates: 1727 - 1788
  Roles: artist
  External IDs: http://viaf.org/viaf/29542582; http://vocab.getty.edu/ulan/500115200; https://rkd.nl/artists/29966; https://www.wikidata.org/entity/Q192720

Candidate 4:
Person: Mary Mohl
  PID: 0QQA-0001-0000-0000
  Names: Mary Mohl; Mohl, Mary

Candidate 5:
Person: Ma